# model-clinic Quickstart

**model-clinic** is a diagnostic and treatment tool for PyTorch models. Think of it as a doctor for neural networks — it examines your model weights, finds problems, prescribes fixes, and applies them safely with rollback support.

This notebook walks through the core workflow:
1. Load a model (we use a tiny synthetic model so nothing downloads)
2. Run `diagnose()` to find issues
3. Compute the health score
4. Generate an HTML diagnostic report

**Requirements**: `pip install model-clinic`

No GPU or real checkpoints needed — all examples use synthetic models.

In [ ]:
# Verify installation
import model_clinic
print(f"model-clinic version: {model_clinic.__version__}")

## 1. Load a Model

model-clinic works with any PyTorch state dict. In real usage you'd load from a `.pt` checkpoint or HuggingFace Hub. Here we use the built-in synthetic model generator to create a toy model with known problems.

In [ ]:
from model_clinic import make_everything_broken, SYNTHETIC_MODELS

# See all available synthetic presets
print("Available synthetic model presets:")
for name in sorted(SYNTHETIC_MODELS.keys()):
    print(f"  {name}")

In [ ]:
# Create a model with every type of problem — ideal for exploring the detector
state_dict = make_everything_broken(hidden=256, layers=6)

print(f"State dict contains {len(state_dict)} tensors")
total_params = sum(t.numel() for t in state_dict.values())
print(f"Total parameters: {total_params:,}")

# Peek at the parameter names
for name, tensor in list(state_dict.items())[:8]:
    print(f"  {name:45s}  shape={str(tensor.shape):20s}  dtype={tensor.dtype}")

## 2. Diagnose

`diagnose()` runs all static detectors on the state dict and returns a list of `Finding` objects. Each finding describes a specific problem on a specific parameter.

Static analysis requires no GPU and no forward pass — it examines the raw weight tensors.

In [ ]:
from model_clinic import diagnose

findings = diagnose(state_dict)

print(f"Found {len(findings)} issue(s)")
print()

In [ ]:
# Print every finding
for f in findings:
    print(f"[{f.severity:5s}] {f.condition:30s}  {f.param_name}")
    if f.details:
        for k, v in f.details.items():
            print(f"          {k}: {v}")

### Finding severities

| Severity | Meaning |
|----------|---------|
| `ERROR`  | Critical — likely causing generation failures, NaN outputs, or complete collapse |
| `WARN`   | Degraded quality — model works but performance is impaired |
| `INFO`   | Advisory — worth knowing about but unlikely to be causing problems |

In [ ]:
# Group findings by severity
from collections import Counter

severity_counts = Counter(f.severity for f in findings)
condition_counts = Counter(f.condition for f in findings)

print("By severity:")
for sev, count in sorted(severity_counts.items()):
    print(f"  {sev}: {count}")

print("\nBy condition:")
for cond, count in sorted(condition_counts.items()):
    print(f"  {cond}: {count}")

## 3. Compute the Health Score

`compute_health_score()` aggregates all findings into a single 0–100 score with a letter grade, plus category breakdowns. The categories are:

- **weights** — basic weight health (NaN/Inf, dead neurons, norms)
- **stability** — training-stability indicators (heavy tails, gates, saturation)
- **output** — output-layer health (token collapse, lm_head issues)
- **activations** — activation-path health (LayerNorm drift, representation issues)

In [ ]:
from model_clinic import compute_health_score, print_health_score

health = compute_health_score(findings)

print(f"Overall score : {health.overall}/100  ({health.grade})")
print(f"Summary       : {health.summary}")
print()
print("Category breakdown:")
for category, score in health.categories.items():
    bar = '#' * (score // 5) + '.' * (20 - score // 5)
    print(f"  {category:15s}  [{bar}]  {score}/100")

In [ ]:
# print_health_score renders the formatted CLI-style output
print_health_score(health)

## 4. Prescribe Treatments

`prescribe()` maps each finding to a recommended treatment. Prescriptions include a risk level (`low`, `medium`, `high`) and an explanation of what the fix does.

In [ ]:
from model_clinic import prescribe

# Conservative mode: only low-risk fixes
prescriptions = prescribe(findings, conservative=True)

print(f"Prescriptions (conservative mode): {len(prescriptions)}")
for rx in prescriptions:
    print(f"  [{rx.risk:6s}] {rx.name:35s}  {rx.description}")
    if rx.explanation:
        print(f"           Why: {rx.explanation[:100]}..." if len(rx.explanation) > 100 else f"           Why: {rx.explanation}")

In [ ]:
# Without conservative mode — all prescriptions, including medium/high risk
all_prescriptions = prescribe(findings, conservative=False)
print(f"All prescriptions: {len(all_prescriptions)}")

risk_counts = Counter(rx.risk for rx in all_prescriptions)
for risk, count in sorted(risk_counts.items()):
    print(f"  {risk}: {count}")

## 5. Generate an HTML Report

The HTML report is the richest view of model health. It includes:
- Health score gauge
- Weight distribution histograms per layer
- LayerNorm drift bar charts
- Dead neuron heatmaps
- Suggested fixes with explanations

All visualizations are inline SVG — no external dependencies, no images folder.

In [ ]:
# Save to file (for viewing in a browser)
import tempfile
import os
from model_clinic import build_meta

# You can also run this from the CLI:
#   model-clinic report checkpoint.pt --output report.html

# For a quick programmatic report, use the report module directly
try:
    from model_clinic._report import generate_html_report
    meta = build_meta(state_dict)
    html = generate_html_report(state_dict, findings, health, meta, prescriptions)

    report_path = os.path.join(tempfile.gettempdir(), "model_clinic_quickstart.html")
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Report saved to: {report_path}")
    print(f"Open it in a browser to see weight distribution histograms and the health score gauge.")
except ImportError:
    print("HTML report module not available in this build. Use: model-clinic report model.pt")

## 6. Compare Against a Healthy Model

For reference, here's what a healthy model looks like. Scores should be close to 100/A.

In [ ]:
from model_clinic import make_healthy_mlp

healthy_sd = make_healthy_mlp(hidden=256, layers=6)
healthy_findings = diagnose(healthy_sd)
healthy_health = compute_health_score(healthy_findings)

print("=== HEALTHY MODEL ===")
print(f"Findings: {len(healthy_findings)}")
print(f"Score:    {healthy_health.overall}/100 ({healthy_health.grade})")
print()

print("=== BROKEN MODEL ===")
print(f"Findings: {len(findings)}")
print(f"Score:    {health.overall}/100 ({health.grade})")

## 7. Using the CLI

Everything above is also available as one-liner CLI commands. Run these in a terminal (not a notebook cell) after `pip install model-clinic`:

```bash
# Examine a real checkpoint
model-clinic exam checkpoint.pt

# Examine a HuggingFace model
model-clinic exam Qwen/Qwen2.5-0.5B-Instruct --hf

# Generate HTML report
model-clinic report checkpoint.pt --output report.html

# Demo on a synthetic broken model (no checkpoint needed)
model-clinic demo everything-broken
model-clinic demo --list

# JSON output for CI pipelines
model-clinic exam checkpoint.pt --json
```

## Next Steps

- **Notebook 02** — Treatment Guide: apply fixes and measure improvement
- **Notebook 03** — Training Monitor: detect problems during training
- `docs/treatment_recipes.md` — Proven fix combinations for common scenarios
- `docs/troubleshooting.md` — When things don't work as expected